Modelo inicial de imagenes (CNN)

In [6]:
# archivo: train_cnn_animals.py
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

In [7]:
# Generar datos
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_dir = os.path.join(BASE_DIR, "train")
valid_dir = os.path.join(BASE_DIR, "valid")
test_dir = os.path.join(BASE_DIR, "test")

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
val_gen = val_datagen.flow_from_directory(
    valid_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_gen.num_classes
print("Número de clases:", num_classes)


Found 699 images belonging to 4 classes.
Found 199 images belonging to 4 classes.
Número de clases: 4


In [8]:
#Config
DATASET = "cat"  # "cat" o "dog"
BASE_DIR = "cat_skin_disease_split" if DATASET=="cat" else "dog_skin_disease_split"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 30
MODEL_SAVE = f"{DATASET}_cnn_model.h5"


In [9]:
#Modelo CNN
def create_cnn(input_shape=(224,224,3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2,2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs, outputs)
    return model

input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)
model = create_cnn(input_shape, num_classes)
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,784,644 (98.36 MB)

 Trainable params: 25,784,644 (98.36 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
#Compilar
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [11]:
#Callbacks
es = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
chk = callbacks.ModelCheckpoint(MODEL_SAVE, monitor='val_loss', save_best_only=True)


In [12]:
#Entrenamiento 
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=[es, chk]
)


Epoch 1/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 459ms/step - accuracy: 0.2430 - loss: 1.5770

22/22 ━━━━━━━━━━━━━━━━━━━━ 14s 551ms/step - accuracy: 0.2532 - loss: 1.4874 - val_accuracy: 0.4020 - val_loss: 1.3436
Epoch 2/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 581ms/step - accuracy: 0.3095 - loss: 1.3633

22/22 ━━━━━━━━━━━━━━━━━━━━ 15s 668ms/step - accuracy: 0.3276 - loss: 1.3526 - val_accuracy: 0.4523 - val_loss: 1.2589
Epoch 3/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 526ms/step - accuracy: 0.4223 - loss: 1.2936

22/22 ━━━━━━━━━━━━━━━━━━━━ 13s 608ms/step - accuracy: 0.4177 - loss: 1.2852 - val_accuracy: 0.5025 - val_loss: 1.1816
Epoch 4/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 521ms/step - accuracy: 0.4229 - loss: 1.2704

22/22 ━━━━━━━━━━━━━━━━━━━━ 13s 600ms/step - accuracy: 0.4406 - loss: 1.2547 - val_accuracy: 0.5126 - val_loss: 1.1455
Epoch 5/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 531ms/step - accuracy: 0.4854 - loss: 1.1956

...

22/22 ━━━━━━━━━━━━━━━━━━━━ 14s 630ms/step - accuracy: 0.6338 - loss: 0.9351 - val_accuracy: 0.6181 - val_loss: 0.9049
Epoch 21/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 14s 632ms/step - accuracy: 0.6252 - loss: 0.9423 - val_accuracy: 0.6131 - val_loss: 0.9061
Epoch 22/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 555ms/step - accuracy: 0.6099 - loss: 0.9051

22/22 ━━━━━━━━━━━━━━━━━━━━ 14s 643ms/step - accuracy: 0.6094 - loss: 0.9229 - val_accuracy: 0.6231 - val_loss: 0.8780
Epoch 23/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 13s 595ms/step - accuracy: 0.6180 - loss: 0.9184 - val_accuracy: 0.6181 - val_loss: 0.8838
Epoch 24/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 601ms/step - accuracy: 0.6787 - loss: 0.8551

22/22 ━━━━━━━━━━━━━━━━━━━━ 15s 695ms/step - accuracy: 0.6495 - loss: 0.8892 - val_accuracy: 0.6432 - val_loss: 0.8520
Epoch 25/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 614ms/step - accuracy: 0.6557 - loss: 0.8921

22/22 ━━━━━━━━━━━━━━━━━━━━ 15s 708ms/step - accuracy: 0.6767 - loss: 0.8640 - val_accuracy: 0.6231 - val_loss: 0.8473
Epoch 26/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 504ms/step - accuracy: 0.6762 - loss: 0.8343

22/22 ━━━━━━━━━━━━━━━━━━━━ 18s 599ms/step - accuracy: 0.6795 - loss: 0.8454 - val_accuracy: 0.6533 - val_loss: 0.8150
Epoch 27/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 14s 616ms/step - accuracy: 0.6652 - loss: 0.8411 - val_accuracy: 0.6482 - val_loss: 0.8279
Epoch 28/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 13s 578ms/step - accuracy: 0.6795 - loss: 0.8391 - val_accuracy: 0.6533 - val_loss: 0.8513
Epoch 29/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 12s 554ms/step - accuracy: 0.6652 - loss: 0.8304 - val_accuracy: 0.6633 - val_loss: 0.8302
Epoch 30/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 12s 563ms/step - accuracy: 0.6938 - loss: 0.8040 - val_accuracy: 0.6633 - val_loss: 0.8719


In [13]:
#Evaluar test
if os.path.isdir(test_dir):
    test_datagen = ImageDataGenerator(rescale=1./255)
    test_gen = test_datagen.flow_from_directory(
        test_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    results = model.evaluate(test_gen)
    print("Resultados en Test:", results)

# Guardar historial de entrenamiento
import json
with open(f"{DATASET}_history.json", "w") as f:
    json.dump({k: [float(x) for x in v] for k,v in history.history.items()}, f)

Found 101 images belonging to 4 classes.
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 93ms/step - accuracy: 0.7228 - loss: 0.8424 
Resultados en Test: [0.8424453735351562, 0.7227723002433777]
